# Lección 14.5 — Casos famosos: AlphaGo, robótica y RLHF

> **Módulo 14 · Aprendizaje por refuerzo (introducción conceptual)** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 60 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lecciones 14.1–14.4; Lección 13.5 (cómo se entrena un LLM)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué ingredientes hicieron posible **AlphaGo** y **AlphaZero**: búsqueda, redes y **autojuego** (*self-play*).
- Explicar el reto de llevar RL a **robots reales**: del simulador al mundo (*sim-to-real*) y la aleatorización de dominio.
- Describir las tres etapas del **RLHF** y construir un **modelo de recompensa** a partir de preferencias humanas.
- Reconocer la **sobreoptimización** de la recompensa (ley de Goodhart) y decidir cuándo RL es buena idea en un proyecto real.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 Una línea de tiempo de hitos

| Año | Hito | Qué aprendió | Ingrediente clave |
|---|---|---|---|
| 1992 | **TD-Gammon** (IBM) | Backgammon a nivel de campeón | Jugar contra sí mismo + una pequeña red |
| 2015 | **DQN** (DeepMind) | 49 juegos de Atari desde los píxeles | Red Q + memoria de repetición (Lección 14.3) |
| 2016 | **AlphaGo** (DeepMind) | Vence 4–1 a Lee Sedol, campeón mundial de Go | Búsqueda + redes + partidas humanas + autojuego |
| 2017 | **AlphaGo Zero / AlphaZero** | Go, ajedrez y shogi **sin datos humanos** | Solo autojuego desde cero |
| 2016 | **Enfriamiento de centros de datos** (Google/DeepMind) | Reducir la energía de enfriamiento hasta ~40 % | RL con datos de sensores reales |
| 2019 | **OpenAI Five** y **AlphaStar** | Dota 2 y StarCraft II a nivel profesional | PPO a escala gigante, autojuego entre poblaciones de agentes |
| 2019 | **Mano robótica de OpenAI** | Resolver un cubo Rubik con una mano robótica | Entrenar solo en simulación + aleatorización de dominio |
| 2022 | **InstructGPT → ChatGPT** | Seguir instrucciones y conversar de forma útil | **RLHF**: recompensa aprendida de preferencias humanas + PPO |
| 2024–25 | **Modelos de "razonamiento"** (OpenAI o1, DeepSeek-R1…) | Resolver problemas de matemáticas y código paso a paso | RL con recompensas **verificables** (¿la respuesta es correcta? ¿pasa los tests?) |

### ✋ Ejercicio 1 — ¿Qué tienen en común?
Mira la columna "ingrediente clave". En casi todos los casos aparece al menos una de estas tres cosas: **(1) un simulador o juego donde equivocarse es gratis**, **(2) una recompensa clara y medible**, **(3) muchísimas partidas**. ¿Cuál de los hitos te parece que **no** tenía un simulador perfecto y por qué fue más difícil?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Los más difíciles son los del **mundo real**:
- **La mano robótica**: la física de una mano real (fricción, desgaste, sensores) no es idéntica a la del simulador. Hubo que entrenar con miles de variaciones (Sección 3).
- **El centro de datos**: no puedes "probar" estrategias de enfriamiento al azar en un edificio real sin arriesgar los servidores. Se usaron datos históricos y muchas medidas de seguridad.
- **ChatGPT**: no hay una fórmula para "respuesta útil": la recompensa tuvo que **aprenderse** a partir de humanos (Sección 4).

En los juegos, en cambio, el simulador es perfecto (las reglas son exactas) y se pueden jugar millones de partidas. Por eso los grandes éxitos del RL empezaron por los juegos.
</details>

---
## 2. 📖 AlphaGo y AlphaZero: pensar hacia adelante y jugar contra uno mismo

El Go tiene más posiciones posibles que átomos en el universo, así que ninguna tabla Q sirve (Lección 14.3). AlphaGo combinó tres ideas:

```
 1. RED DE POLÍTICA  → "¿qué jugadas parecen prometedoras aquí?"     (intuición de un experto)
 2. RED DE VALOR     → "¿quién va ganando en esta posición?"         (un crítico, Lección 14.4)
 3. BÚSQUEDA (MCTS)  → "imaginemos qué pasaría si juego esto, y él responde aquello, y..."
```

La **búsqueda de árbol de Monte Carlo** (MCTS) explora mentalmente las jugadas más prometedoras unos cuantos movimientos hacia adelante, como un buen jugador de ajedrez que "calcula variantes". Las redes le dicen **dónde** mirar y **cuándo** parar.

> 🎯 **La jugada 37:** en la segunda partida contra Lee Sedol, AlphaGo hizo una jugada que los comentaristas creyeron un error: según sus propios cálculos, un humano la habría jugado con una probabilidad de 1 en 10,000. Resultó ser brillante. **Había descubierto algo que los humanos no conocían.**

Vamos a ver estas ideas en un juego mucho más pequeño.

### 📖 El juego de Nim 🥢
Hay **15 palitos** en la mesa. Por turnos, cada jugador quita **1, 2 o 3**. **Gana quien se lleva el último.**

### ✋ Ejercicio 2 — Juega en tu cabeza
1. Si quedan **4** palitos y es tu turno, ¿puedes ganar?
2. Si quedan **5**, ¿cuántos quitas?
3. ¿Ves algún patrón?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No**, si el rival juega bien: quites 1, 2 o 3, le dejas 3, 2 o 1, y se lleva todos.
2. Quitas **1** y le dejas 4 al rival: ahora es él quien está perdido.
3. **Deja siempre un múltiplo de 4** a tu rival. Con 15 palitos, quita 3 (deja 12). Con 10, quita 2 (deja 8).

¿Podrá una máquina descubrir esta regla **sin que nadie se la diga**?
</details>

### 📖 Idea 1: "imaginar" partidas al azar (Monte Carlo)
La forma más simple de evaluar una jugada: imaginar muchas partidas en las que, después de esa jugada, **los dos jugadores juegan al azar**, y contar cuántas ganas. Es la semilla de MCTS.

### 🔮 Ejercicio 3 — ¿Basta con imaginar al azar?
Con **10** palitos la jugada correcta es quitar **2** (dejar 8). Simularemos 2,000 partidas al azar después de cada jugada posible. ¿Crees que las simulaciones elegirán el 2?

✍️ *Mi predicción:*

In [ ]:
PALITOS = 15

def imaginar(palitos, quito, rng, simulaciones=2000):
    """Quito 'quito' palitos y luego imagino partidas en las que ambos juegan al azar."""
    ganadas = 0
    for _ in range(simulaciones):
        quedan, me_toca = palitos - quito, False
        if quedan == 0:
            ganadas += 1; continue
        while True:
            quedan -= rng.integers(1, min(3, quedan) + 1)
            if quedan == 0: break
            me_toca = not me_toca
        ganadas += (not me_toca)      # el último en jugar se llevó el último palito
    return ganadas / simulaciones

rng = np.random.default_rng(0)
for palitos in [10, 7]:          # ✏️ prueba otros valores
    tasas = {q: imaginar(palitos, q, rng) for q in (1, 2, 3)}
    print(f"Quedan {palitos}: % de victorias imaginadas al quitar 1, 2, 3 → "
          + "  ".join(f"{q}: {t:.0%}" for q, t in tasas.items()))

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** Con **10** palitos las tres jugadas salen parecidas (≈ 46–52 %) y el 2, la correcta, ¡queda **última**! Con **7** es todavía peor: la jugada correcta (quitar 3 y dejar 4) sale con **35 %**, la peor de las tres. ¿Por qué? Porque en las partidas imaginadas **yo también juego al azar**: dejarle 4 al rival solo sirve si después sé rematar, y un jugador al azar no sabe. Imaginar partidas en que ambos juegan al azar es una mala aproximación de un rival (y de uno mismo) que juega bien.

Por eso AlphaGo no imaginaba partidas al azar: usaba **redes** para jugar las partidas imaginadas de forma razonable y para **evaluar** posiciones sin llegar hasta el final. Y esas redes se entrenaron… jugando contra sí mismas.
</details>

### 📖 Idea 2: autojuego (*self-play*)
AlphaZero empezó **sin saber nada** y jugó millones de partidas **contra sí mismo**. Cada partida le enseñaba algo a las dos versiones a la vez, y como el rival mejoraba al mismo ritmo, siempre tenía un oponente "a su altura".

Hagamos lo mismo en Nim con Q-learning (Lección 14.3). El truco: **lo que es bueno para mí es malo para mi rival**, así que el valor de dejarle al rival una posición es **menos** el mejor valor que él puede obtener desde ahí:

```
si me llevo el último palito:  objetivo = +1
si no:                         objetivo = − (mejor Q del rival en la posición que le dejo)
```

In [ ]:
def autojuego(partidas, alfa=0.1, epsilon=0.2, semilla=0):
    rng = np.random.default_rng(semilla)
    Q = np.zeros((PALITOS + 1, 4))                  # Q[palitos que quedan, cuántos quito]
    for _ in range(partidas):
        quedan = PALITOS
        while quedan > 0:                           # los dos jugadores usan (y mejoran) la MISMA tabla
            posibles = [q for q in (1, 2, 3) if q <= quedan]
            if rng.random() < epsilon:
                q = rng.choice(posibles)
            else:
                q = max(posibles, key=lambda x: Q[quedan, x])
            if q == quedan:
                objetivo = 1.0                                       # ¡me llevé el último!
            else:
                objetivo = -max(Q[quedan - q, x] for x in (1, 2, 3) if x <= quedan - q)
            Q[quedan, q] += alfa * (objetivo - Q[quedan, q])
            quedan -= q
    return Q

def estrategia(Q):
    return {n: max([q for q in (1, 2, 3) if q <= n], key=lambda x: Q[n, x]) for n in range(1, PALITOS + 1)}

def aciertos(Q):
    e = estrategia(Q)
    ganables = [n for n in range(1, PALITOS + 1) if n % 4 != 0]   # posiciones donde se puede ganar
    return sum(e[n] == n % 4 for n in ganables), len(ganables)

### 🔮 Ejercicio 4 — ¿Cuántas partidas necesita?
Hay 12 posiciones "ganables" (las que no son múltiplo de 4). ¿Cuántas partidas de autojuego crees que necesita para jugar perfecto en todas: 5, 20, 50 o miles?

✍️ *Mi apuesta:*

In [ ]:
for partidas in [5, 20, 50, 1000]:      # ✏️
    Q = autojuego(partidas)
    ok, total = aciertos(Q)
    print(f"{partidas:>5} partidas → juega perfecto en {ok:>2} de {total} posiciones ganables")

Q_nim = autojuego(1000)
print("\nCuántos quita en cada posición:", estrategia(Q_nim))

<details><summary><b>👀 Ver respuesta</b></summary>

- **5 partidas:** acierta en la mitad (≈ 6 de 12): ya domina el final (1, 2, 3 palitos), pero no el principio.
- **20:** ≈ 8 de 12: el conocimiento se va propagando **desde el final hacia atrás**, como el valor del diamante en la Lección 14.3.
- **50 o más:** **12 de 12**. Descubrió solo la regla "deja múltiplos de 4": con 15 quita 3, con 10 quita 2, con 5 quita 1.

En las posiciones múltiplo de 4 da igual qué haga (todas pierden contra un rival perfecto). Nim es pequeñísimo comparado con el Go, pero la receta de AlphaZero es la misma: **jugar contra uno mismo y aprender de cada partida**.
</details>

---
## 3. 📖 Robótica: del simulador al mundo real

Entrenar un robot real con RL tiene tres problemas:
1. **Es lento:** un robot real hace un intento cada pocos segundos; un simulador, miles por segundo.
2. **Es peligroso y caro:** explorar al azar (Lección 14.2) significa chocar, caer, romper cosas.
3. **El simulador nunca es perfecto:** la fricción, el peso, los motores y las cámaras reales son un poco distintos. Un agente entrenado en un simulador perfecto puede fallar estrepitosamente en el mundo real. Es la **brecha simulador-realidad** (*sim-to-real gap*).

La solución más famosa es la **aleatorización de dominio** (*domain randomization*): en cada episodio del simulador se cambian **al azar** la fricción, los pesos, los colores, la iluminación, los retrasos de los motores… El agente aprende una política que funciona **en todos esos mundos**, y el mundo real es "solo uno más".

> 🧩 La mano robótica de OpenAI que resolvió el cubo Rubik (2019) se entrenó **solo en simulación**, con el equivalente a unos **10,000 años** de experiencia, cambiando constantemente el tamaño del cubo, la fricción de los dedos y hasta la gravedad.

### ✋ Ejercicio 5 — El robot de la cocina
Entrenas en un simulador un brazo que debe tomar tazas. En el simulador todas las tazas pesan 300 g, son blancas y la luz es siempre igual. En la cocina real, el robot falla con tazas oscuras y con tazas pesadas. Propón **tres** cosas que aleatorizarías en el simulador.

✍️ *Mis propuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

Por ejemplo:
1. **Peso** de la taza (100–600 g) y su **fricción** (cerámica, vidrio mojado).
2. **Color y textura** de la taza y de la mesa, e **iluminación** (posición, intensidad, sombras).
3. **Forma y tamaño** (con o sin asa), **ruido en la cámara** y pequeños **retrasos** en los motores.

La idea: si el robot nunca vio variedad, aprende "atajos" del simulador (por ejemplo, "una taza es una mancha blanca"). Es el mismo problema del sobreajuste (Lección 4.4): **aprender detalles del conjunto de entrenamiento que no se cumplen en el mundo real**. La aleatorización de dominio es, en el fondo, *data augmentation* (Lección 9.8) para RL.
</details>

---
## 4. 📖 RLHF: cuando la recompensa la dan las personas

¿Cuál es la recompensa de "una buena respuesta de un chatbot"? No hay fórmula. Pero a las personas **sí** se les da bien **comparar**: *"esta respuesta es mejor que aquella"*. El **RLHF** (*Reinforcement Learning from Human Feedback*, Lección 13.5) aprovecha eso en tres etapas:

```
 ETAPA 1: Ajuste supervisado (SFT)
   El LLM pre-entrenado aprende de ejemplos escritos por personas  →  modelo "educado"

 ETAPA 2: Modelo de recompensa
   Personas comparan pares de respuestas: "A es mejor que B"
   Un modelo aprende a dar una CALIFICACIÓN que reproduzca esas preferencias

 ETAPA 3: RL con PPO (Lección 14.4)
   El LLM (la política) genera respuestas, el modelo de recompensa las califica,
   PPO ajusta el LLM para obtener mejores calificaciones…
   …pero con una penalización si se aleja demasiado del modelo de la etapa 1 (penalización KL)
```

Variantes modernas: **RLAIF** (otra IA hace las comparaciones, guiada por principios escritos), **DPO** (*Direct Preference Optimization*: aprende de las preferencias directamente, sin modelo de recompensa ni PPO) y el RL con **recompensas verificables** de los modelos de razonamiento (la recompensa es "¿la respuesta de matemáticas es correcta?").

### ✋ Ejercicio 6 — Conecta con el módulo
Para la **etapa 3** del RLHF, identifica las piezas de la Lección 14.1:
- ¿Quién es el **agente**? ¿Cuál es la **acción**? ¿Qué es la **recompensa**?
- ¿Por qué hace falta la **penalización KL**? (Pista: Lección 14.1, Sección 4.)

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Agente:** el LLM. **Acción:** generar una respuesta (técnicamente, cada palabra o *token* es una acción). **Estado:** la conversación hasta ese momento. **Recompensa:** la calificación del modelo de recompensa.
- **Penalización KL:** el modelo de recompensa es solo una **aproximación** de lo que les gusta a las personas, y el LLM optimiza exactamente esa aproximación. Sin freno, encuentra sus puntos débiles (*reward hacking*): respuestas larguísimas, halagos, frases que "suenan bien" pero no dicen nada. La penalización le dice: "mejora, pero sin dejar de parecerte al modelo original". Es la misma filosofía del recorte de PPO.

¿Lo de las respuestas larguísimas es real? Lo vas a comprobar tú mismo en la práctica integradora.
</details>

---
## 5. 📖 ¿Cuándo usar RL en un proyecto real?

Después de tantos éxitos, es tentador usar RL para todo. Pero la mayoría de los problemas de negocio se resuelven **mejor y más barato** con aprendizaje supervisado. Esta lista de verificación ayuda a decidir:

| Pregunta | Si la respuesta es "sí"… |
|---|---|
| ¿Hay **decisiones en secuencia**, donde lo que hago hoy cambia lo que podré hacer mañana? | Punto para RL |
| ¿Tengo un **simulador** fiable, o puedo permitirme muchos errores en el mundo real? | Punto para RL |
| ¿Puedo escribir una **recompensa** que mida de verdad lo que quiero (sin atajos)? | Punto para RL |
| ¿Tengo ejemplos con **la respuesta correcta**? | Usa **supervisado** (Módulos 4–7) |
| ¿Es una sola decisión independiente con retroalimentación inmediata (qué anuncio mostrar)? | Usa un **bandido** (Lección 14.2), no RL completo |
| ¿Los errores durante el aprendizaje son **peligrosos** o **irreversibles**? | Mucho cuidado: simulador, RL *offline*, supervisión humana |

### ✋ Ejercicio 7 — ¿RL, bandido o supervisado?
Para cada propuesta, recomienda **RL completo**, **bandido** o **supervisado**:
1. Predecir qué clientes dejarán de pagar su préstamo, con 10 años de historial.
2. Elegir qué color de botón "Comprar" mostrar a cada visitante.
3. Controlar los semáforos de una ciudad para reducir el tráfico, con un simulador de tráfico calibrado.
4. Decidir cada día cuánto inventario pedir de 500 productos, sabiendo que lo que sobra hoy queda en la bodega para mañana.

✍️ *Mis recomendaciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Supervisado** (clasificación): tienes la respuesta correcta en el historial (pagó / no pagó). Es el proyecto del Módulo 5.
2. **Bandido**: cada visita es una decisión independiente con respuesta inmediata (clic / no clic). Ver la práctica integradora de la Lección 14.2.
3. **RL completo** es un buen candidato: decisiones encadenadas (cada semáforo afecta al siguiente cruce y al tráfico de dentro de 10 minutos), simulador disponible y una recompensa medible (tiempo total de espera). Aun así, se probaría con mucho cuidado antes de ponerlo en las calles.
4. **RL** es tentador (lo que pido hoy afecta el inventario de mañana), pero en la práctica muchas empresas obtienen casi todo el beneficio con **pronósticos supervisados** de demanda + reglas clásicas de inventario. Empezar por ahí es más barato y más fácil de explicar; RL tendría sentido como segunda etapa, con un buen simulador.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El agente es excelente en el simulador y malo en la realidad | Brecha simulador-realidad | Aleatorización de dominio, calibrar el simulador, ajuste fino con datos reales |
| El agente de autojuego le gana a sí mismo pero pierde con estrategias "raras" | Solo vio rivales parecidos a él | Jugar contra una **población** de versiones pasadas y estrategias variadas |
| El LLM entrenado con RLHF da respuestas larguísimas o aduladoras | Sobreoptimización del modelo de recompensa | Penalización KL, mejores datos de preferencias, detenerse antes |
| El proyecto de RL lleva 6 meses y no supera a una regla simple | RL no era la herramienta adecuada | Comparar siempre contra una **línea base** simple (Lección 4.7) |

### 🔮 Ejercicio 8 — La ley de Goodhart
El economista Charles Goodhart dijo: *"Cuando una medida se convierte en objetivo, deja de ser una buena medida"*. Da un ejemplo de la vida diaria (escuela, trabajo, redes sociales) y relaciónalo con el *reward hacking*.

✍️ *Mi ejemplo:*

<details><summary><b>👀 Ver respuesta</b></summary>

Algunos ejemplos:
- Una escuela que se evalúa **solo** por el resultado de un examen estandarizado termina "enseñando para el examen" en lugar de enseñar a pensar.
- Un centro de llamadas que premia **llamadas cortas** consigue que los agentes cuelguen sin resolver el problema.
- Una red social que optimiza el **tiempo en pantalla** puede terminar promoviendo contenido que indigna o engancha, aunque no le haga bien a nadie.

En todos los casos la medida (examen, duración, tiempo) era una **aproximación** razonable de lo que se quería, hasta que alguien (o algo) la optimizó con mucha fuerza. Exactamente lo que le pasa a un agente de RL con una recompensa imperfecta.
</details>

---
## 7. 📋 Ficha resumen: recetas de los grandes éxitos

| Caso | Política | Recompensa | Ingrediente especial | Lección |
|---|---|---|---|---|
| Atari (DQN) | Red Q | Puntos del juego | Memoria de repetición, red objetivo | 14.3 |
| AlphaGo / AlphaZero | Red de política + red de valor | Ganar (+1) / perder (−1) | Búsqueda MCTS + autojuego | 14.5 |
| Robótica | Red entrenada con PPO u otros | Tarea cumplida, sin caídas | Simulación + aleatorización de dominio | 14.4–14.5 |
| ChatGPT (RLHF) | El propio LLM | Modelo de recompensa aprendido de preferencias | PPO + penalización KL | 13.5, 14.4 |
| Modelos de razonamiento | El propio LLM | ¿Respuesta correcta? ¿Pasa los tests? | Recompensas verificables | 13.5 |

## 8. 🧠 Resumen en 5 puntos
1. Los grandes éxitos del RL empezaron en **juegos**, donde hay un simulador perfecto, una recompensa clara y millones de partidas.
2. **AlphaGo/AlphaZero** combinan redes de política y valor con **búsqueda**, y aprenden jugando **contra sí mismos**.
3. En **robótica** el reto es la brecha **simulador-realidad**; la **aleatorización de dominio** entrena para muchos mundos a la vez.
4. **RLHF** aprende la recompensa de **comparaciones humanas** y luego optimiza el LLM con PPO y una penalización para no alejarse demasiado.
5. Optimizar con demasiada fuerza una recompensa imperfecta la **rompe** (Goodhart); y muchas veces un modelo supervisado o un bandido basta.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué distinguió a AlphaGo Zero de AlphaGo?
- a) Usó más partidas humanas · b) Aprendió solo con autojuego, sin datos humanos · c) No usaba redes neuronales · d) Jugaba ajedrez

**2.** La aleatorización de dominio sirve para…
- a) Explorar más acciones · b) Que la política funcione en un rango amplio de condiciones, incluido el mundo real · c) Acelerar el simulador · d) Reducir γ

**3.** En RLHF, las personas…
- a) Escriben la función de recompensa en Python · b) Comparan pares de respuestas y un modelo aprende de esas comparaciones · c) Juegan contra el LLM · d) Eligen el valor de ε

**4.** Un LLM entrenado con RLHF empieza a dar respuestas cada vez más largas sin ser más útiles. Lo más probable es…
- a) Falta de exploración · b) Sobreoptimización del modelo de recompensa · c) γ demasiado alto · d) Fuga de datos

**5.** Una tienda quiere decidir qué banner mostrar en su página. ¿Qué herramienta recomendarías primero?
- a) AlphaZero · b) PPO con un simulador de clientes · c) Un bandido multibrazo · d) RLHF

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)**
2. **b)**
3. **b)**
4. **b)** Lo comprobarás en la práctica integradora.
5. **c)** Decisión independiente, retroalimentación inmediata (Lección 14.2).
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** En el autojuego de Nim, ambos jugadores usaban **la misma tabla**. ¿Qué ventaja tiene eso? ¿Qué riesgo tendría si el juego tuviera estrategias "de piedra, papel o tijera" (A le gana a B, B a C y C a A)?

**B. Caso de análisis 💼** Una empresa de reparto quiere entrenar con RL a sus drones para aterrizar en patios. Tiene un simulador, pero los patios reales tienen ropa tendida, perros y viento. Propón un plan en 3 etapas, de menor a mayor riesgo.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Ventaja: cada partida enseña a los dos lados a la vez, y el rival siempre está "a tu altura": ni demasiado fácil ni imposible. Riesgo: en juegos tipo piedra-papel-tijera, el agente puede **dar vueltas en círculo** (aprende A, luego B que le gana a A, luego C que le gana a B, luego vuelve a A) sin mejorar de verdad, u olvidar cómo ganarle a estrategias viejas. AlphaStar lo resolvió entrenando contra una **liga** de muchas versiones pasadas y "especialistas" en explotar debilidades.

**B.** Un plan posible:
1. **Simulador con aleatorización de dominio:** viento variable, objetos al azar (ropa, mascotas), distintos tamaños de patio, ruido en sensores. Medir la tasa de éxito y de "casi accidentes".
2. **Pruebas reales controladas:** patios de prueba propios, con red de seguridad, un piloto humano que puede tomar el control y una política de seguridad que **prohíbe** acciones peligrosas (por ejemplo, descender con viento > X km/h) aunque el agente las quiera.
3. **Despliegue gradual:** pocos patios reales, monitoreo de cada aterrizaje, botón de abortar, y usar los datos reales para mejorar el simulador. Volverás a estas ideas de despliegue gradual y monitoreo en el Módulo 15.
</details>

### 9.3 🧩 Práctica integradora: "RLHF en miniatura"

Vas a construir las etapas 2 y 3 del RLHF con un ejemplo de juguete. Cada respuesta de un chatbot se describe con tres números:

| Característica | Qué mide | Rango |
|---|---|---|
| `utilidad` | Qué tanto resuelve la pregunta | 0 a 1 |
| `cortesia` | Qué tan amable es | 0 a 1 |
| `largo` | Longitud (en "párrafos") | 0 a ~8 |

Dos detalles realistas:
- Un poco de largo ayuda (explicar más), pero las respuestas **demasiado** largas se vuelven menos útiles (se pierde el punto).
- Las personas que comparan respuestas tienen un **sesgo**: tienden a preferir las largas ("parece más completa").

Lo harás en 5 pasos cortos. **No avances sin revisar cada respuesta.**

**Paso 1 ▶️ — Las respuestas y las comparaciones humanas**

In [ ]:
def generar_respuestas(n, rng):
    largo = rng.exponential(1.0, n)
    utilidad = np.clip(rng.normal(0.4, 0.15, n) + 0.2 * np.minimum(largo, 1.5)
                       - 0.4 * np.maximum(largo - 1.5, 0), 0, 1)     # demasiado largo → menos útil
    cortesia = rng.uniform(0, 1, n)
    return np.column_stack([utilidad, cortesia, largo])

PESOS_HUMANOS = np.array([3.0, 1.0, 1.0])   # lo que de verdad pesa en las comparaciones (incluye el sesgo por largo)
PESOS_CALIDAD = np.array([3.0, 1.0, 0.0])   # lo que de verdad queremos: utilidad y cortesía, ¡no el largo!

rng = np.random.default_rng(0)
A, B = generar_respuestas(2000, rng), generar_respuestas(2000, rng)
prob_prefiere_A = 1 / (1 + np.exp(-(A - B) @ PESOS_HUMANOS))
prefiere_A = (rng.random(2000) < prob_prefiere_A).astype(int)      # 1 = la persona eligió A

print(f"Comparaciones recogidas: {len(prefiere_A)} · eligieron A el {prefiere_A.mean():.0%} de las veces")
pd.DataFrame(A[:3], columns=["utilidad", "cortesia", "largo"]).round(2)

**Paso 2 🔮 — El modelo de recompensa.** Entrenamos una regresión logística (Lección 5.2) que, mirando la **diferencia** entre A y B, predice si la persona prefirió A. Sus coeficientes son la "recompensa" que asigna a cada característica.

Antes de ejecutar: ¿qué peso crees que le dará al `largo`: cero, positivo o negativo?

✍️ *Mi predicción:*

In [ ]:
modelo_recompensa = LogisticRegression(fit_intercept=False).fit(A - B, prefiere_A)
pesos_rm = modelo_recompensa.coef_[0]
print("Pesos aprendidos por el modelo de recompensa:",
      dict(zip(["utilidad", "cortesia", "largo"], pesos_rm.round(2))))

def recompensa(R):  return R @ pesos_rm          # lo que ve el agente
def calidad(R):     return R @ PESOS_CALIDAD     # lo que de verdad queremos (el agente NO lo ve)

<details><summary><b>👀 Ver respuesta</b></summary>

Aprende pesos muy parecidos a los humanos (≈ 3.1 para utilidad, ≈ 1.0 para cortesía) y **≈ 1.0 positivo para el largo**. El modelo de recompensa hizo bien su trabajo: **reproducir lo que la gente prefiere**… incluido su sesgo. No tiene forma de saber que el largo "no debería" contar.
</details>

**Paso 3 🔮 — Optimizar contra el modelo de recompensa.** En lugar de PPO usaremos la forma más simple de optimizar: ***best-of-N***. El chatbot genera **N** respuestas candidatas y entrega la que el modelo de recompensa califica mejor. Cuanto mayor es N, más fuerte es la optimización.

Antes de ejecutar: al subir N de 1 a 1,024, ¿qué pasará con la **recompensa** (lo que ve el agente) y con la **calidad real**?

✍️ *Mi predicción:*

In [ ]:
def mejor_de_n(N, rng, penalizacion=0.0, repeticiones=300):
    rec, cal, lar = [], [], []
    for _ in range(repeticiones):
        candidatas = generar_respuestas(N, rng)
        puntaje = recompensa(candidatas) - penalizacion * np.abs(candidatas[:, 2] - 1.0)   # (Paso 4)
        elegida = candidatas[np.argmax(puntaje)]
        rec.append(recompensa(elegida)); cal.append(calidad(elegida)); lar.append(elegida[2])
    return np.mean(rec), np.mean(cal), np.mean(lar)

rng = np.random.default_rng(1)
valores_N = [1, 2, 4, 8, 16, 64, 256, 1024]
tabla = pd.DataFrame([mejor_de_n(N, rng) for N in valores_N], index=valores_N,
                     columns=["recompensa (lo que ve)", "calidad real", "largo"]).round(2)
tabla.index.name = "N"
print(tabla)

fig, ax = plt.subplots()
ax.plot(valores_N, tabla.iloc[:, 0], "o-", label="Recompensa del modelo")
ax.plot(valores_N, tabla.iloc[:, 1], "s-", label="Calidad real")
ax.set_xscale("log", base=2); ax.set_xlabel("N (fuerza de la optimización)"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Sobreoptimización: la recompensa sube, la calidad no"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- La **recompensa** sube **siempre** (de ≈ 3 a ≈ 8): el agente está "ganando".
- La **calidad real** primero sube un poco (de ≈ 1.9 a ≈ 2.1 con N entre 2 y 16) y luego **se desploma** (≈ 0.6 con N = 1,024).
- El **largo** de la respuesta elegida pasa de ≈ 1 a ≈ 7 párrafos.

Optimizar un poco contra el modelo de recompensa ayuda; optimizar **demasiado** encuentra su punto débil (le encanta lo largo) y lo explota. Es Goodhart en estado puro, y se ha observado en LLMs reales (Gao et al., 2022, *Scaling Laws for Reward Model Overoptimization*).
</details>

**Paso 4 ✏️ — Ponle freno.** En RLHF real se usa una penalización por alejarse del modelo original. Aquí la imitamos castigando que el largo se aleje de lo "normal" (1 párrafo). Prueba `PENALIZACION` = 0, 0.5, 1.0 y 5.0 con N = 1,024.

In [ ]:
PENALIZACION = 0.5     # ✏️ prueba 0, 0.5, 1.0 y 5.0
rng = np.random.default_rng(1)
rec, cal, lar = mejor_de_n(1024, rng, penalizacion=PENALIZACION)
print(f"Penalización {PENALIZACION} → recompensa {rec:.2f} · calidad real {cal:.2f} · largo {lar:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

| Penalización | Recompensa (lo que ve) | Calidad real | Largo |
|---|---|---|---|
| 0 | ≈ 8.3 | ≈ 0.6 | ≈ 7.4 |
| 0.5 | ≈ 7.2 | ≈ 2.5 | ≈ 4.5 |
| 1.0 | ≈ 5.3 | **≈ 3.8** | ≈ 1.4 |
| 5.0 | ≈ 4.7 | ≈ 3.5 | ≈ 1.0 |

La recompensa que ve el agente **baja** con la penalización, pero la calidad real **sube** muchísimo: con 1.0 es incluso mejor que el mejor valor del Paso 3, porque ahora sí podemos usar N = 1,024 para buscar respuestas **útiles** en lugar de largas. Con 5.0 baja un poco: un freno exagerado obliga a respuestas de exactamente 1 párrafo, aunque un poco más de largo ayudaría. Ese es el papel de la penalización KL en el RLHF real: **no dejar que la política se aleje tanto de lo razonable como para explotar los defectos del modelo de recompensa.**
</details>

**Paso 5 ✋ — Informe para el equipo de producto.** Escribe 3 frases para un gerente que no sabe ML explicando qué pasó y qué recomiendas.

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

> «Entrenamos un "juez automático" que imita las preferencias de nuestros evaluadores, pero heredó su debilidad por las respuestas largas. Cuando dejamos que el chatbot optimice sin límites para agradar a ese juez, produce respuestas cada vez más largas y **peores** de verdad, aunque el juez les ponga mejor nota. Recomendamos limitar cuánto puede alejarse el chatbot de su comportamiento original, revisar periódicamente respuestas con evaluadores humanos y corregir el sesgo de longitud en las guías de evaluación.»
</details>

---
## 📚 Para profundizar
- 🎬 **Documental:** *AlphaGo* (2017, disponible gratis en YouTube): la historia del match contra Lee Sedol y la jugada 37.
- 📰 **Artículos:** Silver et al. (2017), *Mastering the game of Go without human knowledge* · OpenAI (2019), *Solving Rubik's Cube with a Robot Hand* · Ouyang et al. (2022), *Training language models to follow instructions with human feedback* (InstructGPT).
- 📰 **Sobreoptimización:** Gao, Schulman y Hilton (2022), *Scaling Laws for Reward Model Overoptimization*.
- 📖 **Lista de casos:** DeepMind, *Specification gaming examples in AI* (una lista enorme de agentes que hicieron "trampa" con su recompensa).

---
⬅️ **Anterior:** [14.4 Gradientes de política y PPO: más de lo que funcionó](14.4-gradientes-de-politica-y-ppo.ipynb) · ➡️ **Siguiente:** [14.6 Proyecto del módulo: El dron repartidor 🚁📦](14.6-proyecto-del-modulo.md)